In [1]:
!pip install mxnet # GluonNLP 사용을 위한 딥러닝 프레임워크(MXNet) 설치. KoBERT 등 일부 사전학습 모델에서 필요
!pip install gluonnlp pandas tqdm # Gluon 기반 NLP 라이브러리( KoBERT 전처리나 모델 불러올 때 자주 사용) + 데이터 처리(pandas) + 진행 표시줄(tqdm) 설치
!pip install sentencepiece # Google에서 만든 토크나이저 라이브러리(KoBERT 등의 모델에서 사용하는 subword 토크나이저 설치), BERT, KoBERT 등에서 사용하는 서브워드 단위 토크나이징(예: WordPiece, BPE)에 필요
!pip install transformers # HuggingFace의 사전학습 모델(BERT, GPT 등) 라이브러리 설치
!pip install torch # PyTorch: 딥러닝 모델 구축 및 학습용 프레임워크 설치

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 836.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 23.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: graphviz
    Found existing installation: graphviz 0.20.3
    Uninstalling graphviz-0.20.3:
      Successfully uninstalled graphviz-0.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.5/344.5 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gluonnlp: filename=gluonnlp-0.10.0-cp311-c

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
import sys
if 'google.colab' in sys.modules:
    !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
    !sudo apt-get -qq -y install fonts-nanum
    import matplotlib.font_manager as fm
    font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
    for fpath in font_files:
        fm.fontManager.addfont(fpath)

Selecting previously unselected package fonts-nanum.
(Reading database ... 126111 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [3]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [4]:
import os
import json
import random
import torch
import torch.nn as nn # 신경망 모델 구성에 필요한 클래스들
import torch.optim as optim # 최적화 알고리즘(옵티마이저) (예: SGD, Adam 등)
from torch.utils.data import Dataset, DataLoader # Dataset 클래스를 상속해 나만의 데이터셋을 만들고, DataLoader로 미니배치 단위로 나누어 학습할 수 있도록 함.
from tqdm import tqdm # 반복문의 진행 상황을 시각적으로 표시해주는 라이브러리
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "monologg/kobigbird-bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)


# ----- 데이터 로딩 함수 -----
def load_data_from_folder(folder_path, label, max_samples=1000):
    data = [] # 결과 데이터를 담을 리스트(최종적으로 (제목, 본문, 레이블) 형태의 튜플들을 저장할 리스트)
    file_list = [f for f in os.listdir(folder_path) if f.endswith(".json")] # 해당 폴더에서 .json으로 끝나는 뉴스 파일들만 선택
    random.shuffle(file_list) # 항상 같은 순서가 아니라 무작위로 섞어서 다양한 데이터가 학습에 들어가도록 함
    file_list = file_list[:max_samples] # 섞은 리스트에서 뉴스 기사를 최대 max_samples 개수만큼만 자름

    for filename in file_list: # JSON 파일을 하나씩 순회하면서 읽음.
        try:
            with open(os.path.join(folder_path, filename), encoding='utf-8') as f: # 폴더 경로 + 파일 이름을 합쳐서 해당 파일을 UTF-8 인코딩으로 오픈
                item = json.load(f)
                title = item['labeledDataInfo']['newTitle'] # 제목 정보 추출
                content = item['sourceDataInfo']['newsContent'] # 본문 내용 추출
                data.append((title, content, label)) # (제목, 본문, 레이블) 튜플을 리스트에 저장(학습에 필요한 데이터를 하나의 튜플로 저장)
        except Exception as e:
            print(f"Error in {filename}: {e}") # 오류 발생 시 해당 파일명을 출력하고 넘어감
    return data

# ----- PyTorch Dataset 정의 -----
# <요약>
# 이 클래스는 "제목+본문+레이블" 형태의 데이터를 -> KoBERT가 학습할 수 있는 숫자 텐서 형태로 변환함.
# DataLoader에 넣으면 배치 단위로 자동으로 불러와줌.
class ClickbaitKoBigBirdDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        title, content, label = self.samples[idx]
        text = f"{title} [SEP] {content}"

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/241k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/492k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/870 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/458M [00:00<?, ?B/s]

Some weights of BigBirdForSequenceClassification were not initialized from the model checkpoint at monologg/kobigbird-bert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# ----- 실행 -----
# 폴더 경로 설정 (낚시성: 0 / 정상: 1)
fake_path = "/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/자율 프로젝트/TL_Part1_Clickbait_Direct_ET"
real_path = "/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/자율 프로젝트/TL_Part1_NonClickbait_Auto_ET"

fake_samples = load_data_from_folder(fake_path, 0, max_samples=3000)
real_samples = load_data_from_folder(real_path, 1, max_samples=3000)

all_samples = fake_samples + real_samples
random.shuffle(all_samples)

print(f"총 샘플 수: {len(all_samples)}")

총 샘플 수: 6000


In [9]:
# 필요 라이브러리 임포트
from sklearn.model_selection import train_test_split       # 학습/검증 데이터 분리 함수
from sklearn.metrics import accuracy_score, f1_score       # 정확도, F1 스코어 계산 함수
from torch.utils.data import DataLoader                    # PyTorch 데이터 로더
from tqdm import tqdm                                      # 진행 상황 표시 (프로그레스 바)
import torch                                               # PyTorch 기본 패키지
import torch.nn as nn                                      # 손실 함수 등을 위한 PyTorch 신경망 모듈
import torch.optim as optim                                # 옵티마이저 모듈

# ----- 하이퍼파라미터 설정 -----
MAX_LEN = 512            # KoBERT 입력 문장의 최대 길이
BATCH_SIZE = 2           # 미니 배치 사이즈
EPOCHS = 3               # 전체 학습 반복 횟수 (에폭 수)
LEARNING_RATE = 2e-5     # AdamW 옵티마이저에 사용될 학습률

# ----- 학습/검증 데이터 분리 -----
train_samples, val_samples = train_test_split(all_samples, test_size=0.2, random_state=42)
# 전체 데이터를 80% 학습용, 20% 검증용으로 무작위 분할 (재현성을 위해 random_state 고정)

# ----- Dataset 및 DataLoader 구성 -----
train_dataset = ClickbaitKoBigBirdDataset(train_samples, tokenizer, MAX_LEN)
val_dataset = ClickbaitKoBigBirdDataset(val_samples, tokenizer, MAX_LEN)
# 사용자 정의 KoBERT 데이터셋 클래스에 따라 전처리된 Dataset 생성

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# 학습용 데이터는 무작위 셔플, 검증용 데이터는 셔플하지 않음

# ----- 손실 함수 및 옵티마이저 설정 -----
criterion = nn.CrossEntropyLoss()
# 다중 클래스 분류 문제에 적합한 손실 함수 (KoBERT는 이진 분류라도 logits으로 처리하므로 CrossEntropy 사용 가능)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# BERT 계열에 적합한 AdamW 옵티마이저 사용

# ----- 학습 및 검증 루프 시작 -----
for epoch in range(EPOCHS):  # 총 EPOCHS만큼 반복
    # ==== 학습 단계 ====
    model.train()  # 모델을 학습 모드로 설정 (dropout, batchnorm 활성화)
    total_train_loss = 0  # 학습 손실 누적 변수 초기화

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Train"):
        # 각 배치마다 학습 수행
        input_ids = batch['input_ids'].to(DEVICE)              # 입력 토큰 ID를 GPU로 전송
        attention_mask = batch['attention_mask'].to(DEVICE)    # Attention 마스크 전송
        labels = batch['label'].to(DEVICE)                     # 정답 레이블 전송

        optimizer.zero_grad()  # 이전 배치의 gradient 초기화
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)  # forward 연산
        loss = outputs.loss  # BERT 모델의 출력 중 loss 추출
        loss.backward()      # 역전파
        optimizer.step()     # 파라미터 업데이트

        total_train_loss += loss.item()  # 손실 누적

    # ==== 검증 단계 ====
    model.eval()  # 모델을 평가 모드로 설정 (dropout, batchnorm 비활성화)
    total_val_loss = 0  # 검증 손실 누적 변수 초기화
    all_preds, all_labels = [], []  # 예측값과 실제값 저장 리스트

    with torch.no_grad():  # 검증 시에는 gradient 계산 비활성화
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} - Validation"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)  # 가장 높은 logit을 가진 클래스를 예측값으로 선택
            all_preds.extend(preds.cpu().numpy())        # GPU에서 CPU로 이동 후 리스트에 저장
            all_labels.extend(labels.cpu().numpy())      # 실제 정답값도 저장

    # ==== 성능 지표 계산 ====
    val_acc = accuracy_score(all_labels, all_preds)  # 정확도 계산
    val_f1 = f1_score(all_labels, all_preds)          # F1-score 계산 (이진 분류 기준)

    # ==== 결과 출력 ====
    print(f"📘 Epoch {epoch+1}")
    print(f"🔹 Train Loss: {total_train_loss:.4f}")  # 에폭별 평균 학습 손실 출력
    print(f"🔹 Val Loss  : {total_val_loss:.4f}")    # 에폭별 평균 검증 손실 출력
    print(f"🔹 Val Acc   : {val_acc:.4f}")           # 검증 정확도 출력
    print(f"🔹 Val F1    : {val_f1:.4f}")            # 검증 F1-score 출력

Epoch 1 - Train:   0%|          | 0/2400 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 16.12 MiB is free. Process 11998 has 14.72 GiB memory in use. Of the allocated memory 14.48 GiB is allocated by PyTorch, and 122.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# 테스트 데이터 경로
test_fake_path = "/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/자율 프로젝트/VL_Part1_Clickbait_Direct_ET"
test_real_path = "/content/drive/MyDrive/AI+X/AI+X선도인재양성프로젝트(중급)/자율 프로젝트/VL_Part1_NonClickbait_Auto_ET"

# 테스트 샘플 로드
test_fake_samples = load_data_from_folder(test_fake_path, 0)
test_real_samples = load_data_from_folder(test_real_path, 1)

test_samples = test_fake_samples + test_real_samples
random.shuffle(test_samples)

print(f"🔍 테스트 샘플 수: {len(test_samples)}")


In [ ]:
test_dataset = ClickbaitKoBigBirdDataset(test_samples, tokenizer, MAX_LEN)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

model.eval() # 평가 모드로 전환
all_preds = [] # 모델의 예측값 저장 리스트
all_labels = [] # 실제 정답 라벨 저장 리스트

with torch.no_grad(): # 평가 시에는 gradient 계산 비활성화 → 속도 ↑, 메모리 ↓
    for batch in tqdm(test_dataloader, desc="Testing"): # test_dataloader: 테스트 데이터셋의 배치들을 순회
        # 각 배치의 입력과 정답 라벨을 장치(GPU or CPU)로 이동
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        outputs = model(input_ids, attention_mask=attention_mask) # 예측 수행
        logits = outputs.logits # 예측 결과의 로짓 (각 클래스에 대한 점수)
        preds = torch.argmax(logits, dim=1) # 가장 점수가 높은 클래스를 예측값으로 선택

        all_preds.extend(preds.cpu().numpy()) # 예측값을 CPU로 옮기고 리스트에 추가
        all_labels.extend(labels.cpu().numpy()) # 실제 라벨도 리스트에 추가

# 결과 출력
print("📊 테스트 Classification Report")
print(classification_report(all_labels, all_preds, target_names=["Clickbait", "Non-clickbait"]))

f1 = f1_score(all_labels, all_preds)
print(f"F1 Score: {f1:.4f}")

# Confusion Matrix
conf_matrix = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Clickbait", "Non-clickbait"],
            yticklabels=["Clickbait", "Non-clickbait"])
plt.title("테스트 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# <요약>
# 테스트 데이터에 대한 예측 결과를 수집하고
# 분류 성능을 정량적으로 평가하고
# 시각화까지 한 번에 해주는 코드.

In [ ]:
# # ----- 실시간 기사 분류 함수 -----
# def predict_clickbait(model, tokenizer, title, content, max_len=512):
#     model.eval()
#     text = f"{title} [SEP] {content}"
#     encoding = tokenizer.encode_plus(
#         text,
#         add_special_tokens=True,
#         max_length=max_len,
#         return_token_type_ids=False,
#         padding='max_length',
#         truncation=True,
#         return_attention_mask=True,
#         return_tensors='pt'
#     )
#     input_ids = encoding['input_ids'].to(DEVICE)
#     attention_mask = encoding['attention_mask'].to(DEVICE)

#     with torch.no_grad():
#         outputs = model(input_ids, attention_mask=attention_mask)
#         logits = outputs.logits
#         predicted_class = torch.argmax(logits, dim=1).item()

#     if predicted_class == 0:
#         return "🚨 낚시성 기사 (Clickbait)"
#     else:
#         return "✅ 일반 기사 (Non-clickbait)"

# # ----- 예시 사용 -----
# title = "이 방법만 알면 연봉이 2배!"
# content = "최근 많은 사람들이 이 직업을 선택하고 있습니다. 그 이유는..."
# result = predict_clickbait(model, tokenizer, title, content)
# print("▶ 분류 결과:", result)
